# Chapter 5 — First RAG System

Interactive companion to `chapters/chapter_05.qmd`. Run Chapter 1's batch extraction first if you haven't:

```bash
python code/chapter_01/read_ddr.py datasets/sample_ddrs/ --batch --out datasets/ddr_text
```

In [ ]:
import sys
from pathlib import Path

# This chapter reuses Chapter 4's embedding code plus Chapter 5's own
# retrieve-then-generate logic, so both folders need to be on the path.
sys.path.append(str(Path.cwd().parent / "code" / "chapter_04"))
sys.path.append(str(Path.cwd().parent / "code" / "chapter_05"))

from sentence_transformers import SentenceTransformer
from semantic_search import MODEL_NAME, load_chunks, embed_texts

# answer_question() is our small RAG pipeline: embed the question, retrieve
# the top_k most similar reports, then hand their text to an LLM to answer
# from. stub_llm_call() is a stand-in "LLM" used throughout this book so the
# examples run without needing an API key -- see the chapter text for how to
# swap in a real LLM call.
from first_rag import answer_question, stub_llm_call

TEXT_DIR = Path.cwd().parent / "datasets" / "ddr_text"
model = SentenceTransformer(MODEL_NAME)
filenames, texts = load_chunks(TEXT_DIR)
embeddings = embed_texts(model, texts)

In [ ]:
question = "What led to the fishing operation on report 50?"

# answer_question() returns a tuple of two values. We don't need the
# generated answer text here, so it's assigned to `_answer` -- a leading
# underscore is a Python convention meaning "intentionally unused" -- and we
# keep `evidence`, the list of report filenames retrieved as support.
_answer, evidence = answer_question(question, model, filenames, texts, embeddings, stub_llm_call, top_k=3)
print("Evidence retrieved:")
for name in evidence:
    print(" ", name)

## Try it yourself

Notice report #50 (`FORGE-16A-78-32_Drilling_050_2020-12-08.txt`) itself is **not** in this list, even though the question is about it. This is the real, verified retrieval behavior of the system you just built — see the chapter's Field Notes for why.

In [ ]:
# `not in` checks that an item is absent from a list -- the opposite of `in`.
# This confirms, for real, that report #50 does NOT make it into the top-3
# evidence even though the question is directly about it.
assert "FORGE-16A-78-32_Drilling_050_2020-12-08.txt" not in evidence
print("Confirmed: report #50 is not in the top-3 evidence for this query.")